# GTX: Global Tabular eXplanations — original implementation

This notebook is the implementation used to obtain the results of the paper *Global Tabular eXplanations for Individual Neurons* (correlations of Table 2, local approximators, selected percentiles and the figures). The code is the authors' original notebook (run on Google Colab); this version only adds comments.

**Annotated by Claude (Anthropic), October 2026**, on request of the authors. Changes with respect to the original notebook:
1. All cell outputs were cleared (they contained images of the datasets).
2. The cells that ran **Linear Explanations** with the code of its authors (https://github.com/Trustworthy-ML-Lab/Linear-Explanations, no license) were removed; they are replaced by a cell that only *reads* the LE results (`le_siglip_*.csv`). The dataset loader that came from that repository (`data_utils.get_data`) was replaced by `torchvision.datasets.ImageFolder`.
3. Comments and section headers were added. No other code was changed.

**Not included**: the LE code, the partitions' id files (`{train,val,test}_ids_*.pt`, from the LE repository), the LE results and the ablation-score code of LE (Table 3 of the paper). The batched re-implementation used for the baselines and the reproduction check is in `batched/`.

**Pipeline.** (1) activations of the layer for every image (`N`); (2) similarity features from a *different* model; (3) k-means reduction of the training set; (4) greedy search of the table (Algorithm 1), measuring the Pearson correlation of `u_weighted(x) = sim(x, x*) · u(x*)` on the validation partition; (5) test correlation, tables and figures.

In [ ]:
# Basic imports (PyTorch / torchvision / PIL).
import torch
import os
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from torchvision import datasets, transforms, models

In [ ]:
# Colab only: mounts Google Drive, where the data of the original runs lived. Not needed elsewhere.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Choose the explained network and the dataset (one pair per run). The paper uses
#   ResNet50 / ImageNet, ViTB16 / ImageNet, ViTL32 / ImageNet, ResNet18 / Places365, VGG16 / Cifar100.
MODEL = "ResNet50"
DATASET = "ImageNet"
#MODEL = "ResNet18"
#DATASET = "Places365"
#MODEL = "VGG16"
#DATASET = "Cifar100"

In [ ]:
# Import the torchvision constructors of the chosen network.
if MODEL == "ResNet50":
    from torchvision.models import resnet50, ResNet50_Weights
elif MODEL == "ViTL32":
    from torchvision.models import vit_l_32, ViT_L_32_Weights
elif MODEL == "ViTB16":
    from torchvision.models import vit_b_16, ViT_B_16_Weights
elif MODEL == "ResNet18":
    from torchvision.models import resnet18, ResNet18_Weights
elif MODEL == "VGG16":
    from torchvision.models import vgg16, VGG16_Weights
else:
    print("Model not supported")

In [ ]:
# Folder (on Google Drive in the original runs) with the data and caches: the train/val/test ids, the cached activations,
# the Linear-Explanations results and the k-means caches. Change it to your own folder.
path = "/content/drive/MyDrive/Research/Computer_Science/Interpretability/Tabular_Explanations"

In [ ]:
# Load the dataset. ImageNet-val is read from the .tar in Drive; Places365 (val, small) and CIFAR-100 (test) come from torchvision.
if DATASET == "ImageNet":
    !mkdir /content/imagenet_val
    !mkdir /content/imagenet_val/dummy
    !tar -xvf /content/drive/MyDrive/Research/Computer_Science/Interpretability/Tabular_Explanations/data/ILSVRC2012_img_val.tar -C /content/imagenet_val/dummy/
    extract_path = "/content/imagenet_val/dummy/"
    filenames = os.listdir(extract_path)
    dataset_name = 'imagenet_val'
    dataset = datasets.ImageFolder('/content/imagenet_val')  # root used by the original runs (was data_utils.get_data)
elif DATASET == "Places365":
    dataset = datasets.Places365(root="/content", split='val', download=True)
elif DATASET == "Cifar100":
    dataset = datasets.CIFAR100(root="/content", train=False, download=True)

In [ ]:
DATASET

In [ ]:
# Train/validation/test partitions: indices into `dataset`. They are the partitions of the Linear Explanations repository
# (Oikarinen & Weng, 2024): the dataset's validation images (ImageNet-val, Places365-val, CIFAR-100 test) re-split 70/10/20.
# These .pt files are NOT included (they come from the LE repository).
train_ids = torch.load(path + f"/data/train_ids_{DATASET}.pt")
val_ids   = torch.load(path + f"/data/val_ids_{DATASET}.pt")
test_ids  = torch.load(path + f"/data/test_ids_{DATASET}.pt")

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
# Load the explained network in eval mode (pretrained torchvision weights, the Places365 ResNet-18 checkpoint, or the CIFAR-100 VGG16 from torch.hub).
if MODEL == "ResNet50":
    weights = ResNet50_Weights.IMAGENET1K_V1
    studied_model = resnet50(weights=weights)
    studied_model.eval()
    studied_model = studied_model.to(device)
elif MODEL == "ViTL32":
    weights = ViT_L_32_Weights.IMAGENET1K_V1 #  .IMAGENET1K_SWAG_E2E_V1
    studied_model = vit_l_32(weights=weights)
    studied_model.eval()
    studied_model = studied_model.to(device)
elif MODEL == "ViTB16":
    weights = ViT_B_16_Weights.IMAGENET1K_V1
    studied_model = vit_b_16(weights=weights)
    studied_model.eval()
    studied_model = studied_model.to(device)
elif MODEL == "ResNet18":
    !wget http://places2.csail.mit.edu/models_places365/resnet18_places365.pth.tar
    studied_model = models.resnet18(num_classes=365)
    checkpoint = torch.load("resnet18_places365.pth.tar", map_location="cpu")
    state_dict = {k.replace("module.",""):v for k,v in checkpoint['state_dict'].items()}
    studied_model.load_state_dict(state_dict)
    studied_model.eval()
    studied_model = studied_model.to(device)
elif MODEL == "VGG16":
    studied_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "{}_{}".format("cifar100", "vgg16_bn"), pretrained=True)
    studied_model.eval()
    studied_model = studied_model.to(device)

In [ ]:
def resnet_imagenet_preprocess():
    target_mean = [0.485, 0.456, 0.406]
    target_std = [0.229, 0.224, 0.225]
    preprocess = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                   transforms.ToTensor(), transforms.Normalize(mean=target_mean, std=target_std)])
    return preprocess

In [ ]:
def get_cifar_preprocess():
    target_mean = [0.5070, 0.4865, 0.4409]
    target_std = [0.2673, 0.2564, 0.2761]
    preprocess = transforms.Compose([transforms.Resize(32), transforms.ToTensor(),
                         transforms.Normalize(mean=target_mean, std=target_std)])
    return preprocess


In [ ]:
# Input preprocessing of each network.
if MODEL == "ResNet50" or MODEL == "ResNet18":
    transform = resnet_imagenet_preprocess()
elif MODEL == "VGG16":
    transform = get_cifar_preprocess()
else: #if MODEL == "ViTL32":
    transform = weights.transforms()

In [ ]:
# pearson_corr(x, y): Pearson correlation between two 1-D tensors (the metric rho of the paper).
def pearson_corr(x, y):
    """
    Correlación de Pearson entre dos tensores 1D.
    """
    x = x.flatten()
    y = y.flatten()
    corr_matrix = torch.corrcoef(torch.stack([x, y]))
    return corr_matrix[0, 1]

def MSE(x,y):
    return torch.mean((x-y)**2)

## Layer and activation function `u`

In [ ]:
studied_model #encoder.layers.encoder_layer_23.mlp

In [ ]:
# Choose the explained layer. ResNet50/ResNet18: layer1..layer4 (spatially averaged); ViT: the GELU output of the MLP of the last block
# (CLS token); VGG16: 'features_43' (last pooling) or 'classifier_4'.
#studied_model.layer4[0].conv2
layer = studied_model.layer1 #studied_model.encoder.layers.encoder_layer_23.mlp[1] #studied_model.layer1[0].relu
layer_name = "layer1"#"encoder.layers.encoder_layer_23.mlp_1" #'layer1_0_relu'

In [ ]:
# N(img): activations of ALL the units of the chosen layer for one image, i.e. the function u(x) of the paper.
#   - convolutional layers: average over the spatial dimensions of each channel;
#   - ViT: the CLS token (out[0, 0, :]);
#   - VGG16: features_43 has a 1x1 map (take [0, :, 0, 0]); classifier_4 is already a vector.
# Hook
def get_activation(name,activation):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

def N(img):
    activation = {}
    handle = layer.register_forward_hook(get_activation(layer_name,activation))
    img_rgb = img.convert('RGB')
    input_tensor = transform(img_rgb).unsqueeze(0).to(device) # Changed .cpu() to .to(device)

    with torch.no_grad():
        _ = studied_model(input_tensor)

    out = activation[layer_name]
    if MODEL == "ResNet50" or MODEL == "ResNet18":
        out = torch.mean(out[0], dim=[1,2])
    elif MODEL == "ViTL32" or MODEL == "ViTB16":
        out = out[0,0,:]
    elif MODEL == "VGG16":
        if layer_name == "classifier_4":
            out = out[0]
        elif layer_name == "features_43":
            out = out[0,:,0,0]
    return out

In [ ]:
layer_name

In [ ]:
for i in val_ids[0:5]:
    img = dataset[i][0]
    out = N(img)
    #out = torch.mean(out)


In [ ]:
out.shape

## Target activations of the whole dataset

`target_activations` [n_images, n_units] is the matrix of the neuron activations of the chosen layer on all the images of the dataset.

In [ ]:
# Runs N on every image of the dataset and stacks the results: target_activations has shape [n_images, n_units].
def get_neuron_activations(N_function, transform, dataset, device):
    activations = []
    for i in tqdm(range(len(dataset))):
        img, _ = dataset[i]
        out = N_function(img)

        activations.append(out)
        torch.cuda.empty_cache()

    return activations#_tensor

In [ ]:
# Target activations of the whole dataset (computed once and cached on disk).
if os.path.exists(path + f"/data/target_activations_{MODEL}_{layer_name}.pt"):
    target_activations = torch.load(path + f"/data/target_activations_{MODEL}_{layer_name}.pt")
else:
    target_activations = get_neuron_activations(
        N_function, transform, dataset, device
    )
    target_activations = torch.stack(target_activations)
    torch.save(target_activations , path + f"/data/target_activations_{MODEL}_{layer_name}.pt")

In [ ]:
target_activations.shape # debe ser [50000,X]

## Linear Explanations (baseline) — results only

The Linear Explanations of every neuron were computed with the code of the LE authors (https://github.com/Trustworthy-ML-Lab/Linear-Explanations; SigLIP setting, `max_length=10`, tolerance 0.02), which is not redistributed here. The next cell only loads their output: a table with one row per neuron and the columns `bias`, `concept{j}`, `weight{j}` (`j < max_length`) and `val correlation`, plus the concept list of the dataset, to print the LE next to the GTX table.

In [ ]:
import numpy as np
import pandas as pd

max_length = 10   # LE: maximum number of concepts per neuron

# dataset_name: key used by the LE repository for the dataset
if DATASET == "Places365":
    dataset_name = "places365_val"
elif DATASET == "Cifar100":
    dataset_name = "cifar100_val"
else:
    dataset_name = "imagenet_val"

# Output of the LE code (not included): one row per neuron
result_df = pd.read_csv(path + f"/data/le_siglip_{MODEL}_{layer_name}.csv")

# Concept set of the dataset (from the LE repository, not included)
concept_set = path + f'/data/concept_sets/combined_concepts_{dataset_name.split("_")[0]}.txt'
with open(concept_set, 'r') as f:
    concept_text = (f.read()).split('\n')

# Tabular explanation (GTX)

Similarity model, nearest-neighbour approximators, k-means reduction, greedy search and evaluation.

In [ ]:
# Libraries for the tabular explanation (k-means, plotting).
from sklearn.cluster import KMeans
import torch.nn.functional as F
from typing import Tuple, Optional
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
# Similarity model. To avoid leakage it is always DIFFERENT from the explained network:
#   - explained ViT-B/16 or ViT-L/32  -> torchvision ResNet-101 (we use its output vector, the 1000 logits);
#   - explained ResNet / VGG16        -> Hugging Face ViT-B/16 (CLS token of the last hidden state, 768-d).
# Similarity between two inputs is the cosine of these representations.
if MODEL == "ViTL32" or MODEL == "ViTB16":
    from torchvision.models import resnet101, ResNet101_Weights
    from torchvision import transforms
    model_name = "resnet101"
    weights = ResNet101_Weights.DEFAULT
    feature_extractor = weights.transforms()
    feature_model = resnet101(weights=weights)
    feature_model = feature_model.to("cuda")
    feature_model.eval()
else:
    from transformers import ViTFeatureExtractor, ViTModel
    model_name = "google/vit-base-patch16-224"
    feature_extractor = ViTFeatureExtractor.from_pretrained(model_name)
    feature_model = ViTModel.from_pretrained(model_name)
    feature_model = feature_model.to('cuda')

In [ ]:
# (older, one-image-at-a-time version of the feature extraction; superseded by the batched `extraction` below)
def support_model(image) -> torch.Tensor:
    image_rgb = image.convert('RGB')

    inputs = feature_extractor(images=image_rgb, return_tensors="pt")
    inputs = {k: v.to('cuda') for k, v in inputs.items()}

    with torch.no_grad():
        outputs = feature_model(**inputs)

    return outputs.last_hidden_state[:, 0].cpu().numpy()

In [ ]:
# (older version, superseded)
def extraction(M,ids):
    V_mem = []
    for i in tqdm(range(M)):
        i = ids[i]
        img = dataset[i][0]
        v = support_model(img)
        V_mem.append(v)

    V_mem = torch.tensor(V_mem)

    V_mem = V_mem.view(V_mem.size(0), -1)
    return V_mem

In [ ]:
# Batched feature extraction with the Hugging Face ViT (used when the explained network is convolutional).
def extraction(ids, batch_size=32, device="cuda"):
    V_mem = []

    for start in tqdm(range(0, len(ids), batch_size)):
        end = start + batch_size
        batch_ids = ids[start:end]

        images = [dataset[i][0].convert("RGB") for i in batch_ids]

        inputs = feature_extractor(images=images, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = feature_model(**inputs)

        cls_batch = outputs.last_hidden_state[:, 0]
        V_mem.append(cls_batch.cpu())

    V_mem = torch.cat(V_mem, dim=0)
    return V_mem

In [ ]:
# Batched feature extraction with torchvision's ResNet-101 (used when the explained network is a ViT; it redefines `extraction`).
if MODEL == "ViTL32" or MODEL == "ViTB16":
    def extraction(ids, batch_size=32, device="cuda"):
        V_mem = []

        for start in tqdm(range(0, len(ids), batch_size)):
            end = start + batch_size
            batch_ids = ids[start:end]

            images_pil = [dataset[i][0].convert("RGB") for i in batch_ids]

            preprocessed_images_tensors = [feature_extractor(img).to(device) for img in images_pil]
            input_tensor_batch = torch.stack(preprocessed_images_tensors)

            with torch.no_grad():
                outputs = feature_model(input_tensor_batch)
                features = outputs.squeeze(-1).squeeze(-1).cpu()

            V_mem.append(features)

        V_mem = torch.cat(V_mem, dim=0)
        return V_mem

In [ ]:
# Cosine-similarity matrix between two sets of feature vectors.
def cosine_similarity_matrix(V_mem, V_inp):

    V_mem_norm = F.normalize(V_mem, p=2, dim=1)
    V_inp_norm = F.normalize(V_inp, p=2, dim=1)

    sim_matrix = torch.matmul(V_mem_norm, V_inp_norm.T)  # [N, M]

    return sim_matrix

In [ ]:
# u_{kappa=1}: activation of the most similar entry of the table (first, simple version; redefined below).
def N_approx(V_mem, V_inp, O_mem):
    V_mem_norm = torch.nn.functional.normalize(V_mem, p=2, dim=1)
    V_inp_norm = torch.nn.functional.normalize(V_inp, p=2, dim=1)
    CS_M = torch.matmul(V_mem_norm, V_inp_norm.T)  # [N, M]

    best_idx = CS_M.argmax(dim=0)  # [M]

    output = O_mem[best_idx]

    return output,best_idx

In [ ]:
# Helper: percentile of a 1-D tensor (used by the exploratory N_approx_theta).
def percentile_threshold(x: torch.Tensor, p: float) -> torch.Tensor:
    if x.ndim != 1:
        raise ValueError("Tensor must be 1D")
    if not (0 <= p <= 100):
        raise ValueError("Percentile not in [0,100]")

    return torch.quantile(x, p / 100.0)

In [ ]:
# Exploratory variant (not used in the paper): inputs whose best similarity is below a threshold/percentile receive the mean activation
# instead of the activation of their nearest neighbour.
def N_approx_theta(V_mem, V_inp, O_mem, theta=-1,percentile=90,mu=-1):
    V_mem_norm = torch.nn.functional.normalize(V_mem, p=2, dim=1)
    V_inp_norm = torch.nn.functional.normalize(V_inp, p=2, dim=1)

    CS_M = torch.matmul(V_mem_norm, V_inp_norm.T)  # [N, M]

    best_vals, best_idx = CS_M.max(dim=0)  # [M], [M]

    o_mean = O_mem.mean(dim=0)

    output = O_mem[best_idx]
    if theta == -1:
        mask = best_vals < percentile_threshold(best_vals, percentile)
    else:
        mask = best_vals < theta
    if mask.any():
        if mu == -1:
            output[mask] = o_mean
        else:
            output[mask] = mu

    return output,mask,best_vals,best_idx

In [ ]:
# N_approxII: Eq. (3) of the paper, u_weighted(x) = sim(x, x*) * u(x*), with x* the most similar entry of the table.
# This is the approximator used by GTX (greedy search, evaluation and figures) and in the 'weighted' columns of the local tables.
def N_approxII(V_mem, V_inp, O_mem, device=device):
    V_mem_norm = torch.nn.functional.normalize(V_mem, p=2, dim=1)
    V_inp_norm = torch.nn.functional.normalize(V_inp, p=2, dim=1)

    CS_M = torch.matmul(V_mem_norm, V_inp_norm.T)  # [N, M]

    best_vals, best_idx = CS_M.max(dim=0)  # [M], [M]

    o_mean = O_mem.mean(dim=0)

    O_mem = O_mem.to(device)
    best_vals = best_vals.to(device)
    best_idx = best_idx.to(device)
    output = O_mem[best_idx]*best_vals


    return output

In [ ]:
# General nearest-neighbour approximator. method='nearest' is u_{kappa=1}; method='kNN' averages the activations of the `topk` nearest entries.
# CAUTION: in kNN the default weighting is `weight='softmax'` (softmax of the similarities divided by their std), not the uniform mean of
# Eq. (2) of the paper; pass weight='uniform' for the plain mean. The kappa=5 column of the local-method table was computed with this default.
def N_approx(
    V_mem: torch.Tensor,
    V_inp: torch.Tensor,
    O_mem: torch.Tensor,
    method: str = "nearest",
    topk: int = 5,
    weight: str = "softmax",
    eps: float = 1e-8,
) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[torch.Tensor]]:

    assert V_mem.dim() == 2 and V_inp.dim() == 2, "V_mem and V_inp must be [N,D] y [M,D]"
    device = V_mem.device
    V_inp = V_inp.to(device)
    O_mem = O_mem.to(device)

    V_mem_norm = F.normalize(V_mem, p=2, dim=1)
    V_inp_norm = F.normalize(V_inp, p=2, dim=1)

    CS_M = torch.matmul(V_mem_norm, V_inp_norm.T)  # [N, M]

    if method == "nearest":

        best_idx = CS_M.argmax(dim=0)
        output = O_mem[best_idx]
        return output

    elif method == "kNN":
        N = V_mem.shape[0]
        k = min(max(1, int(topk)), N)

        values, indices = torch.topk(CS_M, k=k, dim=0, largest=True, sorted=True)  # values: [k, M], indices: [k, M]

        if weight == "softmax":
            weights = F.softmax(values / (values.std(dim=0, unbiased=False).clamp(min=eps) + eps), dim=0)

        elif weight == "normalized":
            v_min = values.min(dim=0, keepdim=True).values
            v_pos = values - v_min + eps
            weights = v_pos / (v_pos.sum(dim=0, keepdim=True) + eps)
        elif weight == "uniform":
            weights = torch.ones_like(values) / k
        else:
            raise ValueError(f"weight desconocido: {weight}")

        k_dim, M = indices.shape
        flat_idx = indices.reshape(-1)
        selected = O_mem[flat_idx]
        out_shape = (k_dim, M) + O_mem.shape[1:]
        selected = selected.reshape(out_shape)
        expand_shape = (k_dim, M) + tuple(1 for _ in range(selected.dim() - 2))
        weights_reshaped = weights.reshape(expand_shape).to(selected.dtype)
        weighted = selected * weights_reshaped
        output = weighted.sum(dim=0)

        return output
    else:
        raise ValueError("method debe ser 'nearest' o 'kNN'")

In [ ]:
# semi_centroids: for each k-means centroid, the index of the TRAINING input most similar to it. This is the reduction step of the paper
# ('from each cluster we select the input most similar to the centroid').
def semi_centroids(V_mem, V_inp):
    V_mem_norm = torch.nn.functional.normalize(V_mem, p=2, dim=1)
    V_inp_norm = torch.nn.functional.normalize(V_inp, p=2, dim=1)
    CS_M = torch.matmul(V_mem_norm, V_inp_norm.T)  # [N, M]

    best_idx = CS_M.argmax(dim=0)  # [M]

    return best_idx

In [ ]:
# Helper: standardization (z-score) of a vector.
def normalize_vector(x):
    mean = x.mean()
    std = x.std(unbiased=True)
    return (x - mean) / std

In [ ]:
# Earlier variant of the reduction (replaces each centroid by the mean activation of its cluster). Not used for the paper's tables.
def cluster_with_target_mean(V_mem, O_mem, k=2000, device=device):
    X = V_mem.detach().cpu().numpy()
    y = O_mem.detach().cpu().numpy()

    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    centroids = kmeans.cluster_centers_

    O_means = []
    for c in range(k):
        mask = (labels == c)
        if mask.sum() > 0:
            O_means.append(y[mask].mean())
        else:
            O_means.append(0.0)

    V_centroids = torch.tensor(centroids, dtype=V_mem.dtype, device=device)
    O_means = torch.tensor(O_means, dtype=O_mem.dtype, device=device)

    return V_centroids, O_means

In [ ]:
# cluster_with_target_meanI: k-means (scikit-learn, k clusters, n_init=10, random_state=42) on the similarity features of the training set.
# This is the preliminary reduction of the paper (k=2000 for ImageNet and Places365, k=500 for CIFAR-100).
def cluster_with_target_meanI(V_mem, k=2000, device=device):
    X = V_mem.detach().cpu().numpy()

    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    centroids = kmeans.cluster_centers_

    V_centroids = torch.tensor(centroids, dtype=V_mem.dtype, device=device)

    return V_centroids, labels

def cluster_with_target_meanII(V_centroids,labels, O_mem, device=device):
    k = labels_f.max().item() + 1
    y = O_mem.detach()#.cpu().numpy()

    O_means = []
    for c in range(k):
        mask = (labels == c)
        if mask.sum() > 0:
            O_means.append(y[mask].mean())
        else:
            O_means.append(0.0)

    O_means = torch.tensor(O_means, dtype=O_mem.dtype, device=device)

    return O_means

In [ ]:
# Exploratory reductions (keep only far / near-to-mean activations). Not used in the paper.
def reduce_by_target_distance(V_mem, O_mem, k=100, mode="far", device="cpu"):
    mu = O_mem.mean()
    distances = torch.abs(O_mem - mu)

    if mode == "far":
        _, idx = torch.topk(distances, k)
    elif mode == "near":
        _, idx = torch.topk(-distances, k)
    else:
        raise ValueError("mode must be 'far' or 'near'")

    V_reduced = V_mem[idx].to(device)
    O_reduced = O_mem[idx].to(device)
    return V_reduced, O_reduced

In [ ]:
# Exploratory helper (similarity threshold by sampling). Not used in the paper.
def sample_similarity_threshold(V_mem, num_samples=1000, percentile=95, device="cpu"):
    N, D = V_mem.shape
    V_mem = V_mem.to(device)

    V_norm = torch.nn.functional.normalize(V_mem, p=2, dim=1)

    idx1 = torch.randint(0, N, (num_samples,), device=device)
    idx2 = torch.randint(0, N, (num_samples,), device=device)

    sims = (V_norm[idx1] * V_norm[idx2]).sum(dim=1)  # [num_samples]

    threshold = torch.quantile(sims, percentile / 100.0)

    return threshold.item()#, sims

In [ ]:
# Exploratory reduction by activation percentile. Not used in the paper.
def filter_by_target_percentile(V_mem, O_mem, percentile=95, device="cpu"):
    mu = O_mem.mean()
    deviations = torch.abs(O_mem - mu)

    threshold = torch.quantile(deviations, percentile / 100.0)

    idx = torch.nonzero(deviations >= threshold, as_tuple=True)[0]

    V_filtered = V_mem[idx].to(device)
    O_filtered = O_mem[idx].to(device)

    return V_filtered, O_filtered

In [ ]:
# Column-wise Pearson correlation; NaN (undefined correlation) is set to 0, as stated in the paper.
def mean_columnwise_corr(y_pred, y_val):
    x = y_pred - y_pred.mean(dim=0)
    y = y_val - y_val.mean(dim=0)

    num = (x * y).sum(dim=0)

    den = torch.sqrt((x.pow(2).sum(dim=0)) * (y.pow(2).sum(dim=0)))

    corr = num / den

    corr = torch.nan_to_num(corr, nan=0.0)

    return corr.mean(), corr

In [ ]:
# Helpers to convert lists of tensors/numbers into tensors (used when saving results).
def clean_tensor_list(mixed_list):
    return torch.tensor([float(x) for x in mixed_list])

def list_to_tensor_list(mixed_list):
    tensor_list = []
    for x in mixed_list:
        if isinstance(x, torch.Tensor):
            tensor_list.append(x)
        else:  # float or int
            tensor_list.append(torch.tensor(x))
    return tensor_list

In [ ]:
# Percentile of selected examples inside the neuron's activation distribution (used for the table of selected percentiles).
def value_percentiles(x: torch.Tensor, indices: torch.Tensor, eps=1e-8):
    sorted_x, _ = torch.sort(x)
    n = len(x)

    subset_vals = x[indices]

    pos = torch.searchsorted(sorted_x, subset_vals, right=True)
    percentiles = pos.float() / (n + eps) * 100.0
    return percentiles

### Evaluation / training: features, k-means, greedy search (Algorithm 1)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# Reload the cached target activations of the dataset.
target_activations = torch.load(path + f"/data/target_activations_{MODEL}_{layer_name}.pt")

In [ ]:
# Similarity features of the train / validation / test partitions, and the neuron activations of each partition.
# X_*: [n_images, feature_dim];  y_*: [n_images, n_units].
X_train = extraction(train_ids)
X_val = extraction(val_ids)
X_test = extraction(test_ids)

y_train = target_activations[train_ids]
y_val   = target_activations[val_ids]
y_test  = target_activations[test_ids]

In [ ]:
X_val = X_val.to(device)
X_test = X_test.to(device)
X_train = X_train.to(device)

y_train = y_train.to(device)
y_val = y_val.to(device)

torch.cuda.empty_cache()

In [ ]:
# k-means on the similarity features of the training set (cached). k = 2000 (ImageNet, Places365), 500 (CIFAR-100).
# X_train_f are the centroids, labels_f the cluster of every training input.
# Modify if needed
if MODEL == "ViTL32" or MODEL == "ViTB16":
    if os.path.exists(path + "/data/X_train_f_k=2000_ViT.pt"):
        X_train_f = torch.load(path + "/data/X_train_f_k=2000_ViT.pt")
        labels_f  = torch.load(path + "/data/labels_f_k=2000_ViT.pt")

    else:
        X_train_f, labels_f = cluster_with_target_meanI(X_train, k=2000)
        torch.save(X_train_f,path + "/data/X_train_f_k=2000_ViT.pt")
        torch.save(torch.tensor(list(labels_f)),path + "/data/labels_f_k=2000_ViT.pt")

elif DATASET == "Places365":
    if os.path.exists(path + f"/data/X_train_f_k=2000_{DATASET}.pt"):
        X_train_f = torch.load(path + f"/data/X_train_f_k=2000_{DATASET}.pt")
        labels_f  = torch.load(path + f"/data/labels_f_k=2000_{DATASET}.pt")

    else:
        X_train_f, labels_f = cluster_with_target_meanI(X_train, k=2000)
        torch.save(X_train_f,path + f"/data/X_train_f_k=2000_{DATASET}.pt")
        torch.save(torch.tensor(list(labels_f)),path + f"/data/labels_f_k=2000_{DATASET}.pt")

elif DATASET == "Cifar100":
    if os.path.exists(path + f"/data/X_train_f_k=500_{DATASET}.pt"):
        X_train_f = torch.load(path + f"/data/X_train_f_k=500_{DATASET}.pt")
        labels_f  = torch.load(path + f"/data/labels_f_k=500_{DATASET}.pt")
    else:
        X_train_f, labels_f = cluster_with_target_meanI(X_train, k=500)
        torch.save(X_train_f,path + f"/data/X_train_f_k=500_{DATASET}.pt")
        torch.save(torch.tensor(list(labels_f)),path + f"/data/labels_f_k=500_{DATASET}.pt")
else:
    if os.path.exists(path + "/data/X_train_f_k=2000.pt"):
        X_train_f = torch.load(path + "/data/X_train_f_k=2000.pt")
        labels_f  = torch.load(path + "/data/labels_f_k=2000.pt")

    else:
        X_train_f, labels_f = cluster_with_target_meanI(X_train, k=2000)
        torch.save(X_train_f,path + "/data/X_train_f_k=2000.pt")
        torch.save(torch.tensor(list(labels_f)),path + "/data/labels_f_k=2000.pt")

X_train_f = X_train_f.to(device)

In [ ]:
# Cluster labels as a tensor on the device.
labels_f = list(labels_f)
labels_f = torch.tensor(labels_f)
labels_f = labels_f.to(device)

In [ ]:
# Replace each centroid by the training input most similar to it: best_idx are indices into the training set, X_train_f their features.
# From now on the candidates of the greedy search are these k inputs (the reduced table D').
best_idx = semi_centroids(X_train,X_train_f)
X_train_f = X_train[best_idx]

In [ ]:
# Resume support: Corrs / Is are saved every 50 neurons.
if os.path.exists(path + f"/data/Corrs_{MODEL}" + layer_name + ".pt"):
    Corrs = torch.load(path + f"/data/Corrs_{MODEL}" + layer_name + ".pt")
    Is = torch.load(path + f"/data/Is_{MODEL}" + layer_name + ".pt")
    Corrs = list(Corrs)
    Is = list(Is)
else:
    Corrs = []
    Is = []

In [ ]:
# MAIN LOOP, for each neuron `idx` of the layer. Corrs[idx] stores, in order, the TEST correlation of:
#   column 0: u_weighted with the whole training set as the table      ('global method 0' in the code; the 'u_weighted' column of the local-method table)
#   column 1: u_{kappa=1} with the whole training set                  ('global method 1')
#   column 2: u_{kappa=5} with the whole training set                  ('global method 2'; softmax-weighted, see the note on N_approx)
#   column 3: u_weighted with the k reduced inputs (clustered)         ('global method 3')
#   columns 4..: GTX with C = 1, 2, ..., 20 examples. This is Algorithm 1 of the paper (the code calls it 'Local method'):
#       the first example maximizes rho on the VALIDATION partition; each further example is the one that, added to the table,
#       gives the highest validation rho (NaN -> 0; examples already chosen get 0). The TEST correlation after each step is stored.
# Is[idx] are the chosen examples as indices into the reduced set (map them to the dataset with best_idx and train_ids, see the last cells).
C = 20
C = C - 1

for idx in tqdm(range(len(Corrs),y_train.shape[1])):
    corrs = []

    # Global method 0
    y_pred = N_approxII(X_train, X_test,y_train[:,idx])
    y_pred = y_pred.to(device)
    rho = pearson_corr(y_pred, y_test[:,idx])
    corrs.append(rho)

    # Global method 1
    y_pred = N_approx(X_train, X_test,y_train[:,idx],method="nearest")
    y_pred = y_pred.to(device)
    rho = pearson_corr(y_pred, y_test[:,idx])
    corrs.append(rho)

    # Global method 2
    y_pred = N_approx(X_train, X_test,y_train[:,idx],method="kNN",topk=5)
    y_pred = y_pred.to(device)
    rho = pearson_corr(y_pred, y_test[:,idx])
    corrs.append(rho)
    #results_test.append(rho)

    # Global method 3
    y_train_f = y_train[best_idx,idx]
    y_pred = N_approxII(X_train_f, X_test,y_train_f)
    y_pred = y_pred.to(device)
    rho = pearson_corr(y_pred, y_test[:,idx])
    corrs.append(rho)

    # Local method
    M = []
    G_val  = []
    G_test = []
    Y_preds = []

    for n in range(y_train_f.shape[0]):
        y_pred = N_approxII(X_train_f[n:n+1], X_val,y_train_f[n:n+1])
        rho = pearson_corr(y_pred, y_val[:,idx])
        Y_preds.append(y_pred)
        M.append(rho)
    M = torch.tensor(M)
    M = torch.nan_to_num(M, nan=0.0)
    G_val.append(torch.max(M).item())
    idx_M = torch.argmax(M)
    I0 = [idx_M.item()]
    y_pred_val = Y_preds[idx_M]
    y_pred_val = y_pred_val.to(device)

    y_pred_test = N_approxII(X_train_f[I0], X_test,y_train_f[I0])
    y_pred_test = y_pred_test.to(device)
    rho_test = pearson_corr(y_pred_test, y_test[:,idx])
    G_test.append(rho_test)

    for iters in range(C):
        M_i = []
        Y_vals = []
        for n in range(y_train_f.shape[0]):
            if n not in I0:
                I = I0 + []
                I.append(n)
                I = torch.tensor(I)
                y_pred_val  = N_approxII(X_train_f[I], X_val,y_train_f[I])
                y_pred_val = y_pred_val.to(device)
                rho_val = pearson_corr(y_pred_val, y_val[:,idx])
                Y_vals.append(y_pred_val)
                M_i.append(rho_val)
            else:
                M_i.append(0)
                Y_vals.append(y_pred_val*0)
        idx_M = torch.argmax(torch.tensor(M_i)).item()
        I0.append(idx_M)
        y_pred_val = Y_vals[idx_M]
        y_pred_test = N_approxII(X_train_f[I0], X_test,y_train_f[I0])
        y_pred_test = y_pred_test.to(device)
        rho_test = pearson_corr(y_pred_test, y_test[:,idx])
        M_i = torch.tensor(M_i)
        M_i = torch.nan_to_num(M_i, nan=0.0)
        G_val.append(max(M_i))
        G_test.append(rho_test)

    corrs = corrs + G_test
    Corrs.append(corrs)
    Is.append(I0)

    if idx % 50 == 50-1:
        Corrs0 = list_to_tensor_list(Corrs)
        Is0 = list_to_tensor_list(Is)
        Corrs0 = torch.stack(Corrs0)
        Is0 = torch.stack(Is0)
        torch.save(Corrs0,path + f"/data/Corrs_{MODEL}" + layer_name + ".pt")
        torch.save(Is0,path + f"/data/Is_{MODEL}" + layer_name + ".pt")

In [ ]:
# Save the results (correlations and chosen examples) to disk.
Corrs0 = list_to_tensor_list(Corrs)
Is0 = list_to_tensor_list(Is)
Corrs0 = torch.stack(Corrs0)
Is0 = torch.stack(Is0)
torch.save(Corrs0,path + f"/data/Corrs_{MODEL}" + layer_name + ".pt")
torch.save(Is0,path + f"/data/Is_{MODEL}" + layer_name + ".pt")

In [ ]:
# Helper: percentile of each selected example in the neuron's activation distribution.
def indices_to_percentiles(tensor, indices):
    values = tensor[indices]
    sorted_tensor, _ = torch.sort(tensor)
    n = len(sorted_tensor)

    percentiles = []
    for idx, val in zip(indices, values):
        # posición relativa en el tensor ordenado
        rank = (sorted_tensor <= val).sum().item()
        percentile = 100.0 * rank / n
        percentiles.append(percentile)

    percentiles = torch.tensor(percentiles)

    return percentiles

In [ ]:
# Mean fraction of selected examples (C = 20) in the lower half of the activation distribution ([0, 0.5] column of the percentile table).
tops = []
for idx in range(y_train.shape[1]):
    y_train_f = y_train[best_idx,idx]
    n_top = (indices_to_percentiles(y_train_f, Is[idx]) <= 50).sum() / len(Is[idx])
    tops.append(n_top)
tops = torch.tensor(tops)
print(tops.mean())

In [ ]:
layer_name

In [ ]:
# Mean fraction of selected examples (C = 20) in the top decile of the activation distribution ([0.9, 1] column).
tops = []
for idx in range(y_train.shape[1]):
    y_train_f = y_train[best_idx,idx]
    n_top = (indices_to_percentiles(y_train_f, Is[idx]) >= 90).sum() / len(Is[idx])
    tops.append(n_top)
tops = torch.tensor(tops)
print(tops.mean())

## Generate explanations (LE and GTX) and figures

In [ ]:
import io

In [ ]:
# Formats the Linear Explanation of a neuron (weights x concepts + bias) from the LE results table `result_df`, only to print it next to the GTX table.
# Needs `max_length` and `concept_text` (defined in the LE-results cell above).
def LE_explanation_df(idx,result_df):
    # Iterate through possible concepts (up to max_length) for the current neuron 'i'
    explanation = ""
    bias = result_df['bias'][idx]
    for j in range(max_length):
        concept_id_from_df = result_df[f'concept{j}'][idx]
        weight_from_df = result_df[f'weight{j}'][idx]

        # Only consider concepts that are actually present (concept_id != 0)
        # and have a non-zero weight.
        # Note: `weight_from_df` could be a numpy.ndarray scalar or an int (0).
        is_weight_zero = (isinstance(weight_from_df, np.ndarray) and weight_from_df.item() == 0.0) or \
                        (isinstance(weight_from_df, (int, float)) and weight_from_df == 0)

        is_weight_zero = is_weight_zero or weight_from_df == 0

        if not is_weight_zero:
            # Get the actual concept string from the global `concept_text` list
            concept_string = concept_text[concept_id_from_df]

            explanation += str(round(weight_from_df,4)) + " " + concept_string + " + "
    return explanation[:-3] + " + " + str(round(bias,4))

In [ ]:
# Random order of the neurons, to show random examples (not cherry-picked).
ordered = torch.arange(y_train.shape[1])
shuffled = ordered[torch.randperm(len(ordered))]

In [ ]:
# Draws the GTX table of a few random neurons: the C=10 chosen training images sorted by activation, with the activation above each image
# (green: above the neuron's mean activation on the test partition, red: below), plus the histogram of the neuron's activations.
# Saves GTX_<model>_<layer>_<neuron>.png and histogram_<model>_<layer>_<neuron>.png (the figures of the paper).
for i in range(4):
    idx = shuffled[i]
    print(idx)
    mu = y_test[:,idx].mean()
    LE_exp = LE_explanation_df(idx.item(),result_df)
    print(LE_exp)


    y_train_f = y_train[best_idx,idx]

    I0 = Is[idx]

    A = []
    B = []

    for i,iota in enumerate(I0[0:10]):
        bi = best_idx[iota]
        img1 = dataset[train_ids[bi]][0]
        #out = N(img1)[0,idx].mean()
        A.append(img1)
        B.append(y_train_f[iota].cpu())

    sorted_pairs = sorted(zip(B, A), key=lambda item: item[0], reverse=True)
    B_sorted, A_sorted = zip(*sorted_pairs)

    A = list(A_sorted)
    B = list(B_sorted)

    fig, axes = plt.subplots(2, 5, figsize=(20, 8))

    axes = axes.flatten()
    for i, ax in enumerate(axes):
        ax.imshow(A[i])
        ax.axis("off")
        color = "green" if B[i] >= mu else "red"
        ax.set_title(f"{B[i]:.3f}", color=color, fontsize=16, fontweight="bold")

    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=300, bbox_inches="tight")
    buf.seek(0)
    panel_img = Image.open(buf)
    plt.savefig(f"GTX_{MODEL}_{layer_name}_{idx}.png")
    plt.show()
    plt.close(fig)

    sns.set(style="white", context="talk")
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.hist(y_train[idx].cpu(), bins=40, color='tab:green', edgecolor='black', alpha=0.8)

    ax.set_xlabel('Neuron activations', fontsize=14)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.xaxis.tick_bottom()
    ax.yaxis.tick_left()

    plt.tight_layout()
    plt.savefig(f"histogram_{MODEL}_{layer_name}_{idx}.png", dpi=300)
    plt.show()
    plt.close(fig)


In [ ]:
# Colab only: releases the runtime when the notebook ends.
# KILL CELL
from google.colab import runtime
runtime.unassign()